In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64

load_dotenv()

True

In [3]:
with open("blood_work.png","rb") as f:
    image_b64=base64.b64encode(f.read()).decode()
image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka'

In [8]:
llm=ChatGroq(model="qwen/qwen3.8-27b",temperature=0)

message=HumanMessage(content=[
    {"type":"image_url","image_url":{"url":f"data:image/png;base64,{image_b64}"}},
    {"type":"text","text":"This is a blood work report.Extract all test results and flag any values outside the normal range."}                        
])

response=llm.invoke([message])
print(response.content)

Based on the blood work report provided for **Rajesh Sharma, Age 48, Male**, dated **May 7, 2026**, here is a breakdown of all test results with flags for values outside the normal range.

---

### ✅ **COMPLETE BLOOD COUNT (CBC)**

| Test                  | Result         | Normal Range       | Status     |
|-----------------------|----------------|--------------------|------------|
| Hemoglobin            | 15.1 g/dL      | 13.5 – 17.5 g/dL   | ✅ Normal  |
| Hematocrit            | 44%            | 41 – 53%           | ✅ Normal  |
| WBC                   | 6.8 x10³/µL    | 4.5 – 11.0 x10³/µL | ✅ Normal  |
| Platelets             | 220 x10³/µL    | 150 – 400 x10³/µL  | ✅ Normal  |

> *Note: The original report has some garbled characters in the normal ranges (e.g., “13.5□□”17.5”), but based on standard medical reference ranges and context, these are interpreted as above.*

---

### ⚠️ **LIPID PANEL** — **Multiple Abnormalities**

| Test                  | Result         | Normal Range 

In [9]:
from langchain.tools import tool
from langchain.agents import create_agent

from langchain_core.tools import tool


@tool
def get_diet_recommendation(condition: str) -> dict:
    """
    Given a health condition, return a suitable diet plan.
    Condition must be one of: normal, high_cholesterol, high_sugar, low_iron.
    """

    diet_plans = {
        "high_cholesterol": {
            "eat": ["fruits","vegetables","whole grains","lean protein","nuts and seeds"],
            "do_not_eat": ["red meat","fried food","full-fat dairy","processed snacks","foods high in saturated fat"]
        },
        "high_sugar": {
            "eat": ["vegetables","whole grains","legumes","nuts","high-fiber foods"],
            "do_not_eat": ["white rice in excess","white sugar","junk food","sugary drinks","sweets"]
        },

        "low_iron": {
            "eat": ["spinach","lentils","chickpeas","beans","sesame seeds","iron-rich foods","vitamin C rich fruits"],
            "do_not_eat": ["tea with meals","coffee with meals","excess processed food"]
        },

        "normal": {
            "eat": ["vegetables","fruits","whole grains","lean protein","nuts and seeds"],
            "do_not_eat": ["excess sugar","processed food","trans fats","excess fried food"]
        }
    }

    return diet_plans.get(condition.lower(),diet_plans["normal"])


In [10]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.

For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.

Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

result = diet_agent.invoke({
    "messages": [
        HumanMessage(content=[
            {"type": "image_url","image_url": {"url": f"data:image/png;base64,{image_b64}"}},
            {"type": "text","text": "Analyse this blood work report and suggest a diet plan."}
        ])
    ]
})

print(result["messages"][-1].content)

### 🩸 Blood Work Analysis

**Patient:** Rajesh Sharma, Age 48, Male
**Date:** May 7, 2026

#### 1. Complete Blood Count (CBC)
| Test | Result | Normal Range | Status |
| :--- | :--- | :--- | :--- |
| Hemoglobin | 15.1 g/dL | 13.5 – 17.5 | ✅ Normal |
| Hematocrit | 44% | 41 – 53% | ✅ Normal |
| WBC | 6.8 x 10³/uL | 4.5 – 11.0 | ✅ Normal |
| Platelets | 220 x 10³/uL | 150 – 400 | ✅ Normal |

#### 2. Lipid Panel
| Test | Result | Normal Range | Status |
| :--- | :--- | :--- | :--- |
| **Total Cholesterol** | **238 mg/dL** | < 200 | ⚠️ **High** |
| **LDL Cholesterol** | **162 mg/dL** | < 100 | ⚠️ **High** |
| **HDL Cholesterol** | **36 mg/dL** | > 40 | ⚠️ **Low** |
| **Triglycerides** | **188 mg/dL** | < 150 | ⚠️ **High** |

#### 3. Metabolic Panel
| Test | Result | Normal Range | Status |
| :--- | :--- | :--- | :--- |
| Glucose (Fasting) | 92 mg/dL | 70 – 99 | ✅ Normal |
| HbA1c | 5.3% | < 5.7% | ✅ Normal |
| Creatinine | 1.0 mg/dL | 0.7 – 1.3 | ✅ Normal |
| eGFR | 82 mL/min | > 60 | ✅ No